In [1]:
# conda env 'vis2'

import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as colors

import netCDF4 as nc
import at3d
import glob
import imageio.v2 as imageio
import os

from util import load_measurements

import warnings
warnings.filterwarnings("ignore")

# # These are some parameters to make figures nice (and big)
# params = {'legend.fontsize': 'x-large',
#          'axes.labelsize': 'x-large',
#          'axes.titlesize': 'x-large',
#          'xtick.labelsize': 'x-large',
#          'ytick.labelsize': 'x-large'}

# plt.rcParams.update(params)

plt.style.use('seaborn-v0_8-talk')


In [2]:
### Preprocess 1 Plots
def plot_misr_data(filename, fig, ax, vmax=0.8, cbar_loc='right'):
    misr_data = xr.load_dataset(filename)
    brf = misr_data.brf.data
    for i in range(brf.shape[0]):
        c = ax[i].imshow(brf[i,:,:], cmap='gray', vmin=0, vmax=vmax, aspect='equal')
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_title('{}\nVZA={:.1f}\nVAA={:.1f}'.format(misr_data.camera.data[i], misr_data.VZA_center.data[i], misr_data.VAA_center.data[i]))
    if(cbar_loc != None):
        plt.colorbar(c, ax=ax[:], shrink=0.7, location=cbar_loc)
    fig.suptitle('Registered to Ellipsoid')

def plot_misr_data_map(filename, fig, ax, vmax=0.5, cbar_loc=None, s=2.0):
    misr_data = xr.load_dataset(filename)
    for i in range(misr_data.dims['camera']):
        title = '{}\nVZA = {:.1f}\nVAA = {:.1f}'.format(misr_data.camera.data[i],
                                                        misr_data.VZA_center.data[i],
                                                        misr_data.VAA_center.data[i])
        c = ax[i].scatter(
            misr_data.lon275.data,
            misr_data.lat275.data,
            s=s,
            c=misr_data.brf.data[i,:,:],
            vmax=0.5,
            marker='s',
            cmap='gray'
        )
        ax[i].set_title(title)
    fig.supxlabel('Longitude')
    fig.supylabel('Latitude')
    fig.suptitle('Registered to Ellipsoid')
    if(cbar_loc != None):
        plt.colorbar(c, ax=ax[:], shrink=0.7, location=cbar_loc, label='brf')

def overlay_sampling(filename, fig, ax):
    misr_data = xr.load_dataset(filename)
    for i in range(misr_data.dims['camera']):
        ax[i].scatter(misr_data.lon17600.data.ravel(), misr_data.lat17600.data.ravel())
        ax[i].scatter(misr_data.center_lon.data, misr_data.center_lat.data)

def plot_sun_view_grometry(filename, ax):
    misr_data = xr.load_dataset(filename)
    ax.scatter(np.radians(misr_data.SAA_center.data), misr_data.SZA_center.data, s=200.0, marker='*', c='y')
    for i in range(len(misr_data.camera.data)):
        ax.text(np.radians(misr_data.VAA_center.data[i]), misr_data.VZA_center.data[i], misr_data.camera.data[i], c='tab:orange')
    ax.set_ylim([0,90])
    ax.set_rticks([30, 60, 90])
    ax.set_rlabel_position(67.5)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title('Sun/View Geometry\n(Direction of photons)')

# ### Preprocess 2 Plots
# def load_measurements(file_name):
#     """
#     Load from netCDF the sensors that were saved using 'save_forward_model'
#     above. Adapted from at3d.util.load_forward_model.
#     """
#     dataset = nc.Dataset(file_name)

#     groups = dataset.groups
#     sensors = groups['sensors'].groups
#     sensor_dict = at3d.containers.SensorsDict()

#     for key, sensor in sensors.items():
#         sensor_list = []
#         for i, image in sensor.groups.items():
#             sensor_dataset = xr.open_dataset(xr.backends.NetCDF4DataStore(dataset[
#                 'sensors/{}/{}'.format(key, i)]))
#             sensor_dataset['stokes'] = (['stokes_index'],sensor_dataset['stokes'].data.astype(bool))
#             sensor_dataset['use_subpixel_rays'] = sensor_dataset['use_subpixel_rays'].data.astype(bool)
#             sensor_dict.add_sensor(key, sensor_dataset)
#     return sensor_dict

def plot_measurements(filename, instrument, fig, ax, s=2, vmin=None, vmax=None, set_title=True):
    sensor_dict = load_measurements(filename)
    sensor_list = sensor_dict[instrument]['sensor_list']
    for i, cam in enumerate(sensor_list):
        z = np.unique(cam.ray_z.data)
        if(len(z) == 1):
            title = '{}\nz={} km'.format(cam.name, z[0])
        else:
            title = '{}\nz=non-uniform'.format(cam.name)
        c = ax[i].scatter(cam.ray_x.data, cam.ray_y.data, s=s, c=cam.I.data*np.pi/cam.mu0, marker='s', cmap='gray', vmin=vmin, vmax=vmax)
        if(set_title):
            ax[i].set_title(title)
    return c

def make_movie(outfile, nframes=None, fps=3, bounce=True):
    images = []
    dir = os.path.dirname(outfile)
    files = np.sort(glob.glob(dir + '/*.png'))
    if nframes != None:
        for i in range(nframes):
            fi = files[i]
            images.append(imageio.imread(fi))
        if(bounce):
            for i in range(nframes-2, 0, -1):
                fi = files[i]
                images.append(imageio.imread(fi))
    else:
        for fi in files:
           images.append(imageio.imread(fi))
    imageio.mimsave(outfile, images, loop=15, duration=3, fps=fps)
    print('Movie saved at {}'.format(outfile))

def measurements_to_gif(filename, instrument, outfile, s=2, vmin=None, vmax=None):
    sensor_dict = load_measurements(filename)
    sensor_list = sensor_dict[instrument]['sensor_list']
    dir = os.path.dirname(outfile)
    os.makedirs(dir, exist_ok=False)
    for i, cam in enumerate(sensor_list):
        z = np.unique(cam.ray_z.data)
        f, ax = plt.subplots(1, 1, figsize = (5, 5))
        ax.scatter(cam.ray_x.data, cam.ray_y.data, s=s, c=cam.I.data*np.pi/cam.mu0, marker='s', cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_xlim([-5, 24])
        ax.set_ylim([-11, 28])
        if(len(z) == 1):
            title = '{}\nz={} km'.format(cam.name, z[0])
        else:
            title = '{}\nz=non-uniform'.format(cam.name)
        ax.set_title(title)
        f.savefig('{}/frame{:03d}.png'.format(dir, i))
        plt.close(f)
    make_movie(outfile, nframes=len(sensor_list), fps=10)

### AT3D Plots
def plot_obs_minus_at3d(obs_filename, at3d_filename, instrument, fig, ax, s=2, vmin=None, vmax=None, set_title=True):
    sensor_dict_obs = load_measurements(obs_filename)
    sensor_list_obs = sensor_dict_obs[instrument]['sensor_list']
    sensor_dict_at3d = load_measurements(at3d_filename)
    sensor_list_at3d = sensor_dict_at3d[instrument]['sensor_list']
    for i, (cam_obs, cam_at3d) in enumerate(zip(sensor_list_obs, sensor_list_at3d)):
        assert cam_obs.name == cam_at3d.name
        assert np.array_equal(cam_obs.ray_x.data, cam_at3d.ray_x.data)
        assert np.array_equal(cam_obs.ray_y.data, cam_at3d.ray_y.data)
        assert np.array_equal(cam_obs.ray_z.data, cam_at3d.ray_z.data)
        brf_obs = cam_obs.I.data*np.pi/cam_obs.mu0
        brf_at3d = cam_at3d.I.data*np.pi/cam_at3d.mu0
        brf_diff = brf_obs - brf_at3d
        z = np.unique(cam_obs.ray_z.data)
        if(len(z) == 1):
            title = '{}\nz={} km'.format(cam_obs.name, z[0])
        else:
            title = '{}\nz=non-uniform'.format(cam_obs.name)
        c = ax[i].scatter(cam_obs.ray_x.data, cam_obs.ray_y.data, s=s, c=brf_diff, marker='s', cmap='RdBu', vmin=vmin, vmax=vmax)
        if(set_title):
            ax[i].set_title(title)
    return c

def brf_diff_movie(measurements_file, iter_dir, out_dir, instrument, s=2, brf_vmin=0, brf_vmax=0.5, diff_range=0.25):
    os.makedirs(out_dir, exist_ok=True)
    sensor_dict_obs = load_measurements(measurements_file)
    sensor_list_obs = sensor_dict_obs[instrument]['sensor_list']
    iter_files = np.sort(glob.glob(iter_dir + 'iter*.nc'))
    for j, iter_file in enumerate(iter_files):
        fig, ax = plt.subplots(3, 9, figsize=(15, 7), sharex=True, sharey=True)
        sensor_dict_at3d = load_measurements(iter_file)
        sensor_list_at3d = sensor_dict_at3d[instrument]['sensor_list']
        for i, (cam_obs, cam_at3d) in enumerate(zip(sensor_list_obs, sensor_list_at3d)):
            assert cam_obs.name == cam_at3d.name
            assert np.array_equal(cam_obs.ray_x.data, cam_at3d.ray_x.data)
            assert np.array_equal(cam_obs.ray_y.data, cam_at3d.ray_y.data)
            assert np.array_equal(cam_obs.ray_z.data, cam_at3d.ray_z.data)
            brf_obs = cam_obs.I.data*np.pi/cam_obs.mu0
            brf_at3d = cam_at3d.I.data*np.pi/cam_at3d.mu0
            brf_diff = brf_obs - brf_at3d
            z = np.unique(cam_obs.ray_z.data)
            if(len(z) == 1):
                title = '{}\nz={} km'.format(cam_obs.name, z[0])
            else:
                title = '{}\nz=non-uniform'.format(cam_obs.name)
            c0 = ax[0, i].scatter(cam_obs.ray_x.data, cam_obs.ray_y.data, s=s, c=brf_obs, marker='s', cmap='gray', vmin=brf_vmin, vmax=brf_vmax)
            c1 = ax[1, i].scatter(cam_obs.ray_x.data, cam_obs.ray_y.data, s=s, c=brf_at3d, marker='s', cmap='gray', vmin=brf_vmin, vmax=brf_vmax)
            c2 = ax[2, i].scatter(cam_obs.ray_x.data, cam_obs.ray_y.data, s=s, c=brf_diff, marker='s', cmap='RdBu', vmin=-diff_range, vmax=diff_range)
            ax[0, i].set_title(title)
        plt.colorbar(c1, ax=ax[0:2,:], location='right', shrink=0.8, label='brf')
        plt.colorbar(c2, ax=ax[2,:], location='right', shrink=0.8, label='obs-at3d')
        sup_title = os.path.basename(iter_file).split('.')[0]
        plt.suptitle(sup_title)
        fig.savefig('{}{}.png'.format(out_dir, sup_title))
        plt.close(fig)
    make_movie(out_dir + 'movie.gif', nframes=None, fps=3, bounce=False)

def optical_depth_movie(iter_dir, out_dir, vmin=None, vmax=None, log=True):
    os.makedirs(out_dir, exist_ok=True)
    iter_files = np.sort(glob.glob(iter_dir + 'iter*.nc'))
    for i, iter_file in enumerate(iter_files):
        fig, ax = plt.subplots(1, 1, figsize=(8,6))
        sensors, solvers, rte_grid = at3d.util.load_forward_model(iter_file)
        if(log):
            solvers[0.67].medium['cloud'].extinction.integrate('z').plot(y='y', ax=ax, norm=colors.LogNorm(), vmin=vmin, vmax=vmax, cbar_kwargs={'label': 'optical depth'})
        else:
            solvers[0.67].medium['cloud'].extinction.integrate('z').plot(y='y', ax=ax, vmin=vmin, vmax=vmax, cbar_kwargs={'label': 'optical depth'})
        title = os.path.basename(iter_file).split('.')[0]
        ax.set_title(title)
        fig.savefig('{}{}.png'.format(out_dir, title))
        plt.close(fig)
    make_movie(out_dir + 'movie.gif', nframes=None, fps=3, bounce=False)

def extinction_cross_section_movie(iter_dir, out_dir, dim='x', loc=1, vmin=None, vmax=None, log=True):
    warnings.filterwarnings("ignore")
    os.makedirs(out_dir, exist_ok=True)
    iter_files = np.sort(glob.glob(iter_dir + 'iter*.nc'))
    for i, iter_file in enumerate(iter_files):
        fig, ax = plt.subplots(1, 1, figsize=(8,6))
        sensors, solvers, rte_grid = at3d.util.load_forward_model(iter_file)
        if(log):
            if(dim=='x'):
                solvers[0.67].medium['cloud'].extinction.sel(x=loc, method='nearest').plot(y='z', ax=ax, norm=colors.LogNorm(), vmin=vmin, vmax=vmax)
            elif(dim=='y'):
                solvers[0.67].medium['cloud'].extinction.sel(y=loc, method='nearest').plot(y='z', ax=ax, norm=colors.LogNorm(), vmin=vmin, vmax=vmax)
        else:
            if(dim=='x'):
                solvers[0.67].medium['cloud'].extinction.sel(x=log, method='nearest').plot(y='z', ax=ax, vmin=vmin, vmax=vmax)
            elif(dim=='y'):
                solvers[0.67].medium['cloud'].extinction.sel(y=log, method='nearest').plot(y='z', ax=ax, vmin=vmin, vmax=vmax)
        sup_title = os.path.basename(iter_file).split('.')[0]
        plt.suptitle(sup_title)
        fig.savefig('{}{}.png'.format(out_dir, sup_title))
        plt.close(fig)
    make_movie(out_dir + 'movie.gif', nframes=None, fps=3, bounce=False)


In [3]:
region_title = 'CAMP2Ex_RF17_A'
rte_grid_title = 'default'
misr_data_file = '../data/MISR_OBS_{}.nc'.format(region_title)
measurements_file = '../data/MISR_OBS_{}_RTG_{}.nc'.format(region_title, rte_grid_title)

measurement_movie_out = './MISR_movies/MISR_{}_RTG_{}/movie.gif'.format(region_title, rte_grid_title)

# Preprocess 1 Plots

In [ ]:
fig, ax = plt.subplots(1, 9, figsize=(25, 4))
plot_misr_data(misr_data_file, fig, ax, vmax=0.5, cbar_loc=None)

In [ ]:
fig, ax = plt.subplots(1, 9, figsize=(25, 4), sharex=True, sharey=True)
plot_misr_data_map(misr_data_file, fig, ax, vmax=0.5, cbar_loc='right', s=2.0)
overlay_sampling(misr_data_file, fig, ax)
fig.subplots_adjust(top=0.73, bottom=0.16, left=0.07, right=0.77)

In [ ]:
f, ax = plt.subplots(1, 1, figsize = (6, 6), subplot_kw={'projection': 'polar'})
plot_sun_view_grometry(misr_data_file, ax)

# Preprocess 2 plots

In [ ]:
fig, ax = plt.subplots(1, 9, figsize=(17, 2.3), sharex=True, sharey=True)
plot_measurements(measurements_file, 'MISR', fig, ax, s=2)

measurements_to_gif(measurements_file, 'MISR', outfile=measurement_movie_out, s=10)

# AT3D Plots

In [4]:
expr_name = 'example_iterations'
iter_dir = '../{}/'.format(expr_name)

In [ ]:
x = 14
out_dir = './{}/extinction_cs_x_{:d}_log_movie/'.format(expr_name, x)
extinction_cross_section_movie(iter_dir, out_dir, dim='x', loc=x, vmin=0.1, vmax=25, log=True)

In [5]:
out_dir = './{}/optical_depth_movie/'.format(expr_name)
optical_depth_movie(iter_dir, out_dir, vmin=0.1, vmax=15, log=True)

Movie saved at ./example_iterations/optical_depth_movie/movie.gif


In [6]:
out_dir = './{}/brf_diff_movie/'.format(expr_name)
brf_diff_movie(measurements_file, iter_dir, out_dir, 'MISR', s=2, brf_vmin=0, brf_vmax=0.5, diff_range=0.25)

Movie saved at ./example_iterations/brf_diff_movie/movie.gif


# 1D vs. 3D Optical Depth

In [ ]:
measurements_file = './measurements_1Dtau.nc'
iter_dir = './iterations03/'

iter_files = np.sort(glob.glob(iter_dir + 'iter*.nc'))
sensors, solvers, rte_grid = at3d.util.load_forward_model(iter_files[-1])

cloud_3D = solvers[0.67].medium['cloud']

measurements = load_measurements(measurements_file)
cloud_1D = measurements['MISR']['sensor_list'][4]


In [ ]:
from scipy.interpolate import griddata

x_1D = cloud_1D.ray_x.data
y_1D = cloud_1D.ray_y.data
tau_1D = cloud_1D.tau.data

x_3D = cloud_3D.x.data
y_3D = cloud_3D.y.data

grid_x, grid_y = np.meshgrid(x_3D, y_3D, indexing='ij')
tau_1D_interpolated = griddata((x_1D, y_1D), tau_1D, (grid_x, grid_y), method='linear')

cloud_3D['tau_3D'] = cloud_3D.extinction.integrate('z')
cloud_3D['tau_1D_interp'] = (['x', 'y'], tau_1D_interpolated)


In [ ]:
vmin = 0
vmax = 16
vrange = 8
f, ax = plt.subplots(1, 3, figsize=(18,5), sharex=True, sharey=True)
cloud_3D.tau_3D.plot(ax=ax[0], y='y', vmin=vmin, vmax=vmax)
cloud_3D.tau_1D_interp.plot(ax=ax[1], y='y', vmin=vmin, vmax=vmax)
(cloud_3D.tau_3D - cloud_3D.tau_1D_interp).plot(ax=ax[2], y='y', vmin=-vrange, vmax=vrange, cmap='RdBu_r')

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8,6))
ax.scatter(cloud_3D.tau_3D.data, cloud_3D.tau_1D_interp.data, s=5)
ax.plot([0,20], [0,20], 'r')
ax.set_xlabel('AT3D Optical Depth')
ax.set_ylabel('1D Optical Depth')